# 📚 Sistema de Gestión Editorial — Installer

**Idempotent:** Run this notebook as many times as you want.
- **First run** → Creates the full project structure in your Drive.
- **Subsequent runs** → Skips existing files/folders. Only updates `Configuracion_Estilos` (APA 7 styles).

---
### Pre-requisite (one-time only)
Enable the **Apps Script API** in your Google Cloud Console:
1. Go to → https://console.cloud.google.com/apis/library/script.googleapis.com
2. Click **Enable**.
3. Come back here and run the cells below.
---

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install --quiet google-auth google-auth-oauthlib google-auth-httplib2 \
    google-api-python-client gspread

In [ ]:
# ── Cell 2: Authentication ───────────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
import gspread

SCOPES = [
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/documents',
    'https://www.googleapis.com/auth/script.projects',
]

creds, _ = google.auth.default(scopes=SCOPES)
drive_service  = build('drive',  'v3', credentials=creds)
sheets_service = build('sheets', 'v4', credentials=creds)
docs_service   = build('docs',   'v1', credentials=creds)
script_service = build('script', 'v1', credentials=creds)
gc             = gspread.authorize(creds)
print('✅ Authenticated successfully.')

In [ ]:
# ── Cell 3: Project configuration — edit here ────────────────────────────────
PROJECT_NAME = 'Tesis_Editorial'

CHAPTERS = [
    {'key': 'cap1', 'title': 'Introduccion',       'tipo': 'FRONT',    'orden': 1, 'mostrar_num': False, 'suma': False, 'formato': 'ROMANO_MIN', 'estilo': 'ITALIC'},
    {'key': 'cap2', 'title': 'Marco_Teorico',      'tipo': 'CHAPTER',  'orden': 2, 'mostrar_num': True,  'suma': True,  'formato': 'ARABIGO',    'estilo': 'BOLD'},
    {'key': 'cap3', 'title': 'Metodologia',        'tipo': 'CHAPTER',  'orden': 3, 'mostrar_num': True,  'suma': True,  'formato': 'ARABIGO',    'estilo': 'BOLD'},
    {'key': 'cap4', 'title': 'Resultados',         'tipo': 'CHAPTER',  'orden': 4, 'mostrar_num': True,  'suma': True,  'formato': 'ARABIGO',    'estilo': 'BOLD'},
    {'key': 'cap5', 'title': 'Conclusiones',       'tipo': 'CHAPTER',  'orden': 5, 'mostrar_num': True,  'suma': True,  'formato': 'ARABIGO',    'estilo': 'BOLD'},
    {'key': 'ane1', 'title': 'Anexo_Instrumentos', 'tipo': 'APPENDIX', 'orden': 6, 'mostrar_num': True,  'suma': True,  'formato': 'LETRA_MAY',  'estilo': 'NORMAL'},
]

In [ ]:
# ── Cell 4: Helper functions ─────────────────────────────────────────────────
def find_drive_item(name, mime_type, parent_id=None):
    q = f"name='{name}' and mimeType='{mime_type}' and trashed=false"
    if parent_id:
        q += f" and '{parent_id}' in parents"
    results = drive_service.files().list(q=q, fields='files(id,name)').execute()
    files = results.get('files', [])
    return files[0] if files else None

def get_or_create_folder(name, parent_id=None):
    existing = find_drive_item(name, 'application/vnd.google-apps.folder', parent_id)
    if existing:
        print(f'  📁 Folder already exists: {name}')
        return existing['id']
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.folder'}
    if parent_id:
        meta['parents'] = [parent_id]
    folder = drive_service.files().create(body=meta, fields='id').execute()
    print(f'  📁 Created folder: {name}')
    return folder['id']

def get_or_create_sheet(name, parent_id):
    existing = find_drive_item(name, 'application/vnd.google-apps.spreadsheet', parent_id)
    if existing:
        print(f'  📊 Sheet already exists: {name}')
        return existing['id'], False
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.spreadsheet', 'parents': [parent_id]}
    sheet = drive_service.files().create(body=meta, fields='id').execute()
    print(f'  📊 Created sheet: {name}')
    return sheet['id'], True

def get_or_create_doc(name, parent_id):
    existing = find_drive_item(name, 'application/vnd.google-apps.document', parent_id)
    if existing:
        print(f'  📄 Doc already exists: {name}')
        return existing['id'], False
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.document', 'parents': [parent_id]}
    doc = drive_service.files().create(body=meta, fields='id').execute()
    print(f'  📄 Created doc: {name}')
    return doc['id'], True

print('✅ Helpers ready.')

In [ ]:
# ── Cell 5: APA 7 default styles (always re-applied) ────────────────────────
APA7_STYLES_HEADER = [
    'ID_Etiqueta','Nombre_Visible','Fuente','Tamano','Interlineado',
    'Negrita','Italica','Sangria_1era','Alineacion','Color_Texto',
    'Espaciado_Antes','Espaciado_Despues','Color_Marcado'
]
APA7_STYLES_ROWS = [
    ['H1_APA',    'Encabezado Nivel 1', 'Times New Roman', 12, 2.0, True,  False, 0,  'CENTER',  '#000000', 0,  0,  '#D9EAD3'],
    ['H2_APA',    'Encabezado Nivel 2', 'Times New Roman', 12, 2.0, True,  False, 0,  'LEFT',    '#000000', 0,  0,  '#D9EAD3'],
    ['H3_APA',    'Encabezado Nivel 3', 'Times New Roman', 12, 2.0, True,  True,  0,  'LEFT',    '#000000', 0,  0,  '#D9EAD3'],
    ['H4_APA',    'Encabezado Nivel 4', 'Times New Roman', 12, 2.0, True,  False, 36, 'LEFT',    '#000000', 0,  0,  '#D9EAD3'],
    ['H5_APA',    'Encabezado Nivel 5', 'Times New Roman', 12, 2.0, True,  True,  36, 'LEFT',    '#000000', 0,  0,  '#D9EAD3'],
    ['TIT_CAP',   'Titulo Capitulo',    'Times New Roman', 12, 2.0, True,  False, 0,  'CENTER',  '#000000', 0,  0,  '#D9EAD3'],
    ['TEXTO_APA', 'Parrafo Normal',     'Times New Roman', 12, 2.0, False, False, 36, 'JUSTIFY', '#000000', 0,  0,  '#FCE5CD'],
    ['FIG_TIT',   'Titulo de Imagen',   'Arial',           10, 1.0, True,  False, 0,  'LEFT',    '#000000', 12, 6,  '#D0E0F0'],
    ['TABLA_TIT', 'Titulo de Tabla',    'Arial',           10, 1.0, True,  False, 0,  'LEFT',    '#000000', 12, 6,  '#EAD1DC'],
]
MAPA_HEADER    = ['ID_Documento','Tipo_Bloque','Orden','Mostrar_Num','Suma_Contador','Formato_Num','Estilo_Num','Titulo_Texto']
REGISTRO_HEADER = ['ID_Doc','ID_Etiqueta','ID_Rango','Texto_Referencia','Prefijo_Calculado','Pagina_Calculada','Estado']
print('✅ APA 7 data ready.')

In [ ]:
# ── Cell 6: Create folder structure ─────────────────────────────────────────
print(f'\n🗂  Setting up "{PROJECT_NAME}"...')
root_id   = get_or_create_folder(PROJECT_NAME)
cap_id    = get_or_create_folder('Capitulos',     parent_id=root_id)
config_id = get_or_create_folder('Configuracion', parent_id=root_id)
output_id = get_or_create_folder('Compilado',     parent_id=root_id)
print('\n✅ Folder structure ready.')

In [ ]:
# ── Cell 7: Create/update Google Sheets ─────────────────────────────────────
print('\n📊 Setting up Editorial_Config sheet...')
sheet_id, sheet_created = get_or_create_sheet('Editorial_Config', config_id)
sh = gc.open_by_key(sheet_id)

def get_or_create_ws(spreadsheet, title):
    try:
        return spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        return spreadsheet.add_worksheet(title=title, rows=100, cols=20)

# Configuracion_Estilos: ALWAYS overwrite
print('  Updating Configuracion_Estilos (APA 7)...')
ws_estilos = get_or_create_ws(sh, 'Configuracion_Estilos')
ws_estilos.clear()
ws_estilos.append_row(APA7_STYLES_HEADER)
for row in APA7_STYLES_ROWS:
    ws_estilos.append_row([str(v) for v in row])
print('  ✅ Configuracion_Estilos updated.')

# Mapa_Estructura: only write headers if new
ws_mapa = get_or_create_ws(sh, 'Mapa_Estructura')
if sheet_created or ws_mapa.cell(1, 1).value != 'ID_Documento':
    ws_mapa.clear()
    ws_mapa.append_row(MAPA_HEADER)
    print('  ✅ Mapa_Estructura headers written.')
else:
    print('  ⏭️  Mapa_Estructura already has data — skipped.')

# Registro_Elementos: only write headers if new
ws_registro = get_or_create_ws(sh, 'Registro_Elementos')
if sheet_created or ws_registro.cell(1, 1).value != 'ID_Doc':
    ws_registro.clear()
    ws_registro.append_row(REGISTRO_HEADER)
    print('  ✅ Registro_Elementos headers written.')
else:
    print('  ⏭️  Registro_Elementos already has data — skipped.')

print(f'\n✅ Sheet: https://docs.google.com/spreadsheets/d/{sheet_id}')

In [ ]:
# ── Cell 8: Create chapter Docs ─────────────────────────────────────────────
print('\n📄 Setting up chapter documents...')
existing_mapa = ws_mapa.get_all_records()
existing_ids  = {r['ID_Documento'] for r in existing_mapa}
doc_registry  = {}

for cap in CHAPTERS:
    doc_name = f"Cap_{cap['orden']:02d}_{cap['title']}"
    doc_id, created = get_or_create_doc(doc_name, cap_id)
    doc_registry[cap['key']] = doc_id

print('\n✅ All chapter docs ready.')

In [ ]:
# ── Cell 9: Register Docs in Mapa_Estructura ────────────────────────────────
print('\n🗺  Registering docs in Mapa_Estructura...')
rows_added = 0
for cap in CHAPTERS:
    doc_id = doc_registry[cap['key']]
    id_doc = f"id_doc_{cap['key']}"
    if id_doc in existing_ids:
        print(f'  ⏭️  {id_doc} already registered — skipped.')
        continue
    row = [id_doc, cap['tipo'], cap['orden'], str(cap['mostrar_num']),
           str(cap['suma']), cap['formato'], cap['estilo'],
           cap['title'].replace('_', ' ')]
    ws_mapa.append_row([str(v) for v in row])
    print(f'  ✅ Registered: {id_doc}')
    rows_added += 1

if rows_added == 0:
    print('  ℹ️  No new rows needed.')
print('\n✅ Mapa_Estructura up to date.')

In [ ]:
# ── Cell 10: Attach Apps Script to each Doc ─────────────────────────────────
import requests, time

GITHUB_RAW = 'https://raw.githubusercontent.com/kestuardoflores-creator/editorial-system/main/apps-script/'

def fetch_source(filename):
    r = requests.get(GITHUB_RAW + filename)
    r.raise_for_status()
    return r.text

def build_script_content(sheet_id):
    sidebar_js    = fetch_source('sidebar.js')
    format_engine = fetch_source('format-engine.js')
    sidebar_html  = fetch_source('sidebar.html')
    init_block = f"""
// Auto-injected by installer
function _autoSetSheetId() {{
  setSheetId('{sheet_id}');
}}
"""
    return {'files': [
        {'name': 'sidebar',       'type': 'SERVER_JS', 'source': sidebar_js + init_block},
        {'name': 'format-engine', 'type': 'SERVER_JS', 'source': format_engine},
        {'name': 'sidebar',       'type': 'HTML',      'source': sidebar_html},
    ]}

def get_or_create_script(doc_id):
    try:
        result = script_service.projects().list(pageSize=50).execute()
        for p in result.get('projects', []):
            if p.get('parentId') == doc_id:
                return p['scriptId'], False
    except Exception:
        pass
    body = {'title': 'Editorial Script', 'parentId': doc_id}
    project = script_service.projects().create(body=body).execute()
    return project['scriptId'], True

print('\n⚙️  Attaching Apps Script to each chapter doc...')
content = build_script_content(sheet_id)
for cap in CHAPTERS:
    doc_id = doc_registry[cap['key']]
    script_id, created = get_or_create_script(doc_id)
    script_service.projects().updateContent(scriptId=script_id, body=content).execute()
    action = 'Created' if created else 'Updated'
    print(f"  ✅ {action}: Cap_{cap['orden']:02d}_{cap['title']}")
    time.sleep(0.5)

print('\n✅ All scripts attached.')

In [ ]:
# ── Cell 11: Summary ────────────────────────────────────────────────────────
print('=' * 60)
print('📚 INSTALLER SUMMARY')
print('=' * 60)
print(f'  Root folder    : https://drive.google.com/drive/folders/{root_id}')
print(f'  Config sheet   : https://docs.google.com/spreadsheets/d/{sheet_id}')
print(f'  Chapters folder: https://drive.google.com/drive/folders/{cap_id}')
print()
print('  Documents registered:')
for cap in CHAPTERS:
    doc_id = doc_registry[cap['key']]
    print(f"    [{cap['orden']}] {cap['title']:30s} -> {doc_id}")
print('=' * 60)
print('✅ Installer finished. No data was lost on re-run.')